# Evaluacion de enfoques de vision para GestCorp 98

Comparamos distintos enfoques para localizar 7 elementos de la interfaz sobre las 3 capturas de `experiments/ground_truth/`, contra el ground truth real (`alta_cliente.json`): un circulo verde marca la posicion real y uno rojo la prediccion de cada enfoque, en imagenes guardadas dentro de `vision_evaluation/<enfoque>/`.

Orden: RapidOCR (sin LLM) -> OmniParser (YOLOv8 + Florence-2) -> UI-TARS-7B-DPO (Hugging Face `transformers`). Los tres dan buenos resultados y son los que nos quedamos para seguir construyendo el agente de vision (ver `PROJECT.md`, fase 2).

Se evaluaron y descartaron aparte (no estan en este notebook):
- `qwen2.5vl:7b` (Ollama): ancla su propio formato de coordenadas y no responde fiable a instrucciones de unidad (pixeles ni porcentaje). Ademas, al llamarlo via la API simple de Ollama no hay forma de saber con certeza a que resolucion interna redimensiono la imagen antes de "verla", así que tampoco se puede corregir el escalado con garantias.
- `Molmo-7B-D` (Hugging Face `transformers`, ~16 GB): no llegamos a evaluarlo, UI-TARS ya dio resultados suficientemente buenos antes.
- `llama3.2-vision:11b`: descartado sin evaluar, falla al cargar en Ollama por un bug conocido y sin resolver ([issue #16547](https://github.com/ollama/ollama/issues/16547)), no algo de este notebook.


In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw

GT_DIR = Path("ground_truth")
GT = json.loads((GT_DIR / "alta_cliente.json").read_text(encoding="utf-8"))
_Path = Path  # alias usado en las celdas de evaluacion (p.ej. _Path(image_name).stem)

# (imagen, nombre_del_target, descripcion para el prompt)
TARGETS = [
    ("01_login.png", "usuario_field", "el campo de texto donde se escribe el nombre de usuario (ya contiene 'admin')"),
    ("01_login.png", "contraseña_field", "el campo de texto vacio donde se escribe la contraseña"),
    ("01_login.png", "entrar_button", "el boton 'Entrar' para iniciar sesion"),
    ("02_alta_cliente.png", "nombre_field", "el campo de texto vacio para escribir el nombre del cliente, en el formulario 'Alta de cliente'"),
    ("02_alta_cliente.png", "nif_field", "el campo de texto vacio para escribir el NIF del cliente"),
    ("02_alta_cliente.png", "anadir_button", "el boton 'Añadir cliente' para guardar el nuevo cliente"),
    ("03_popup_validacion.png", "popup_aceptar_button", "el boton 'Aceptar' del popup de aviso 'Datos incompletos'"),
]

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. RapidOCR (sin LLM)

OCR puro y determinista: detecta todo el texto de la imagen con su caja delimitadora. Primero mostramos la lista completa detectada en una imagen, y despues comparamos contra el ground truth igual que con el resto de enfoques: para los botones buscamos la deteccion cuyo texto coincide con la etiqueta del propio boton (su centro SI es el target real); para los campos vacios, al no haber texto dentro, usamos como aproximacion la etiqueta mas cercana (p.ej. "Usuario:" para `usuario_field`) sabiendo de antemano que su centro no sera el centro exacto del campo -- el circulo rojo frente al verde en cada imagen muestra ese desajuste.

In [5]:
from rapidocr_onnxruntime import RapidOCR

_ocr_engine = RapidOCR()


def ask_rapidocr(image_name: str, target_desc: str | None = None) -> dict:
    image_path = GT_DIR / image_name
    t0 = time.perf_counter()
    result, _ = _ocr_engine(str(image_path))
    elapsed = time.perf_counter() - t0
    detecciones = [
        {"texto": texto, "bbox": box, "confianza": round(float(score), 3)}
        for box, texto, score in (result or [])
    ]
    return {"raw": detecciones, "elapsed_s": elapsed}


In [ ]:
resultado = ask_rapidocr("02_alta_cliente.png")
print(f"tiempo: {resultado['elapsed_s']:.2f}s  -  {len(resultado['raw'])} detecciones\n")
for d in resultado["raw"]:
    print(d["texto"], "->", d["bbox"])


In [7]:
import unicodedata

EVAL_OUT_DIR_RAPIDOCR = _Path("vision_evaluation") / "rapidocr"
EVAL_OUT_DIR_RAPIDOCR.mkdir(parents=True, exist_ok=True)


def _normalize(texto: str) -> str:
    """minusculas y sin tildes/enies, para comparar sin depender de como
    el OCR haya transcrito acentos (p.ej. "contraseña" -> "contrasenia")."""
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return sin_acentos.lower().strip()


# Para cada target, la palabra clave (normalizada) que debe contener el texto
# detectado por el OCR. En los botones el texto esta DENTRO del target
# (coincide con su propia etiqueta); en los campos vacios no hay nada que
# leer ahi, asi que usamos como aproximacion la etiqueta mas cercana (p.ej.
# "Usuario:"), sabiendo de antemano que su centro no sera el centro del
# campo real -- es justo lo que este cuadro va a dejar ver.
TARGET_OCR_KEYWORD = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre:",
    "nif_field": "nif:",
    "anadir_button": "anadir cliente",
    "popup_aceptar_button": "aceptar",
}


def match_ocr_target(detecciones: list[dict], keyword: str):
    """Busca la primera deteccion cuyo texto normalizado contiene `keyword`
    y devuelve el centro de su bbox, o None si no aparece ninguna."""
    for det in detecciones:
        if keyword in _normalize(det["texto"]):
            xs = [p[0] for p in det["bbox"]]
            ys = [p[1] for p in det["bbox"]]
            return sum(xs) / len(xs), sum(ys) / len(ys)
    return None


_rapidocr_cache: dict[str, dict] = {}

filas_rapidocr = []
for image_name, target_key, _desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]

    if image_name not in _rapidocr_cache:
        _rapidocr_cache[image_name] = ask_rapidocr(image_name)
    resultado = _rapidocr_cache[image_name]
    tiempo = resultado["elapsed_s"]

    keyword = TARGET_OCR_KEYWORD[target_key]
    punto = match_ocr_target(resultado["raw"], keyword)
    pred_x = pred_y = error_px = None
    if punto is not None:
        pred_x, pred_y = punto
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    else:
        estado = f"ERROR: ninguna deteccion contiene '{keyword}'"

    filas_rapidocr.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px"
    )

    # circulo verde = posicion real, rojo = punto que da RapidOCR via etiqueta
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR_RAPIDOCR / f"{_Path(image_name).stem}__{target_key}.png")

df_rapidocr = pd.DataFrame(filas_rapidocr)
aciertos = (df_rapidocr["error_px"] < 20).sum()
print(f"\nResumen rapidocr: {aciertos}/{len(df_rapidocr)} aciertos (error < 20px)")
print(f"error medio: {df_rapidocr['error_px'].mean():.1f}px")
print(f"tiempo medio (deteccion completa por imagen): {df_rapidocr['tiempo_s'].mean():.2f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR_RAPIDOCR}/")
df_rapidocr


01_login.png/usuario_field: ok  gt=(272,138) pred=(112.90322303771973,137.82051467895508) error=159.1px
01_login.png/contraseña_field: ok  gt=(272,188) pred=(98.06451225280762,188.4615478515625) error=173.9px
01_login.png/entrar_button: ok  gt=(208,274) pred=(209.03225708007812,275.00001525878906) error=1.4px


02_alta_cliente.png/nombre_field: ok  gt=(219,1399) pred=(60.64515924453735,1400.6410522460938) error=158.4px
02_alta_cliente.png/nif_field: ok  gt=(525,1399) pred=(387.0967559814453,1400.6410522460938) error=137.9px
02_alta_cliente.png/anadir_button: ok  gt=(475,1483) pred=(475.4838562011719,1485.8975219726562) error=2.9px


03_popup_validacion.png/popup_aceptar_button: ok  gt=(1397,888) pred=(1398.0645141601562,889.7436218261719) error=2.0px

Resumen rapidocr: 3/7 aciertos (error < 20px)
error medio: 90.8px
tiempo medio (deteccion completa por imagen): 2.30s
capturas con circulos guardadas en: vision_evaluation\rapidocr/


,imagen,target,gt_x,gt_y,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,112.903223,137.820515,159.096878,2.334270,ok
1,01_login.png,contraseña_field,272,188,98.064512,188.461548,173.936100,2.334270,ok
2,01_login.png,entrar_button,208,274,209.032257,275.000015,1.437214,2.334270,ok
3,02_alta_cliente.png,nombre_field,219,1399,60.645159,1400.641052,158.363344,2.062538,ok
4,02_alta_cliente.png,nif_field,525,1399,387.096756,1400.641052,137.913008,2.062538,ok
5,02_alta_cliente.png,anadir_button,475,1483,475.483856,1485.897522,2.937644,2.062538,ok
6,03_popup_validacion.png,popup_aceptar_button,1397,888,1398.064514,889.743622,2.042892,2.931032,ok


## 2. OmniParser (detector)

`microsoft/OmniParser-v2.0`: detector YOLOv8 de iconos (`ultralytics`) + Florence-2 como captioner (`transformers`). Ya ejecutado (~1.4 GB descargados la primera vez).

**Receta de carga verificada contra el `handler.py` oficial del repo**: el detector sale de `icon_detect/model.pt`; el captioner son solo los PESOS afinados en `icon_caption/` (safetensors), pero el **procesador** (tokenizer + preprocesado de imagen) no viene en ese repo -- hay que cargarlo del modelo base original, `microsoft/Florence-2-base`. Los recortes de cada icono se redimensionan a 64x64 antes de pedir la descripcion (asi lo hace Microsoft en su propio handler).

**Resultado real**: el detector YOLOv8 localiza cajas muy precisas para cada widget (incluso campos vacios, ver `vision_evaluation/omniparser/*__todas_las_cajas.png`). El emparejamiento por palabra clave sobre el *caption* de Florence-2 (celda de mas abajo) falla casi siempre -- Florence-2 esta entrenado para describir fotos, no widgets de UI, y no devuelve las palabras españolas que buscamos. O sea: el fallo no es de deteccion, es de como identificamos que caja es cual. Pendiente (no implementado todavia): emparejar cada caja del detector con el texto mas cercano de RapidOCR en vez de con el caption de Florence-2.

In [8]:
_omniparser_detector = None
_omniparser_caption_model = None
_omniparser_caption_processor = None


def _load_omniparser():
    global _omniparser_detector, _omniparser_caption_model, _omniparser_caption_processor
    if _omniparser_detector is None:
        from huggingface_hub import snapshot_download
        from transformers import AutoModelForCausalLM, AutoProcessor
        from ultralytics import YOLO

        local_dir = snapshot_download("microsoft/OmniParser-v2.0")
        _omniparser_detector = YOLO(f"{local_dir}/icon_detect/model.pt")
        # el procesador se carga del Florence-2 base original: el repo de
        # OmniParser solo publica los pesos afinados del captioner, no su
        # propio procesador (asi lo hace su handler.py oficial)
        _omniparser_caption_processor = AutoProcessor.from_pretrained(
            "microsoft/Florence-2-base", trust_remote_code=True
        )
        _omniparser_caption_model = AutoModelForCausalLM.from_pretrained(
            f"{local_dir}/icon_caption", torch_dtype=torch.float16, trust_remote_code=True
        ).to("cuda")
    return _omniparser_detector, _omniparser_caption_model, _omniparser_caption_processor


def ask_omniparser_elements(image_name: str, target_desc: str | None = None) -> dict:
    detector, caption_model, caption_processor = _load_omniparser()
    image_path = GT_DIR / image_name
    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    t0 = time.perf_counter()
    results = detector(str(image_path), verbose=False)[0]

    crops = []
    cajas = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        crop = image.crop((x1, y1, x2, y2)).resize((64, 64))
        crops.append(crop)
        cajas.append((x1, y1, x2, y2, float(box.conf[0])))

    captions = []
    if crops:
        inputs = caption_processor(
            images=crops, text=["<CAPTION>"] * len(crops), return_tensors="pt", do_resize=False
        ).to(caption_model.device, dtype=torch.float16)
        generated_ids = caption_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=20,
            num_beams=1,
            do_sample=False,
        )
        captions = [
            texto.strip()
            for texto in caption_processor.batch_decode(generated_ids, skip_special_tokens=True)
        ]

    elementos = []
    for (x1, y1, x2, y2, conf), caption in zip(cajas, captions):
        elementos.append(
            {
                "bbox_px": [round(v, 1) for v in (x1, y1, x2, y2)],
                "bbox_norm": [
                    round(x1 / width, 4), round(y1 / height, 4),
                    round(x2 / width, 4), round(y2 / height, 4),
                ],
                "caption": caption,
                "confianza": round(conf, 3),
            }
        )
    elapsed = time.perf_counter() - t0
    return {"raw": elementos, "elapsed_s": elapsed}


In [ ]:
# prueba rapida: dispara la descarga de OmniParser (~1.4 GB) y muestra
# todos los elementos que detecta en la imagen del formulario
resultado = ask_omniparser_elements("02_alta_cliente.png")
print(f"tiempo: {resultado['elapsed_s']:.1f}s  -  {len(resultado['raw'])} elementos\n")
for el in resultado["raw"]:
    print(el)


In [10]:
# Visualizamos TODAS las cajas que detecta OmniParser en cada imagen (no
# solo la que coincide por palabra clave), para comprobar si localiza bien
# los campos/botones aunque el caption no diga el nombre exacto que
# buscamos (p.ej. el boton "Anadir cliente", que Florence-2 transcribe
# mal en el caption pero cuya caja sale casi perfecta).
from PIL import ImageFont

EVAL_OUT_DIR_OMNIPARSER = _Path("vision_evaluation") / "omniparser"
EVAL_OUT_DIR_OMNIPARSER.mkdir(parents=True, exist_ok=True)

_omniparser_cache: dict[str, dict] = {}
_font = ImageFont.load_default(size=28)

for image_name in dict.fromkeys(t[0] for t in TARGETS):
    if image_name not in _omniparser_cache:
        _omniparser_cache[image_name] = ask_omniparser_elements(image_name)
    elementos = _omniparser_cache[image_name]["raw"]

    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    for i, el in enumerate(elementos):
        x1, y1, x2, y2 = el["bbox_px"]
        draw.rectangle((x1, y1, x2, y2), outline=(30, 120, 255), width=4)
        draw.text((x1 + 4, max(y1 - 32, 0)), str(i), fill=(30, 120, 255), font=_font)

    out_path = EVAL_OUT_DIR_OMNIPARSER / f"{_Path(image_name).stem}__todas_las_cajas.png"
    img.save(out_path)
    print(f"{image_name}: {len(elementos)} cajas dibujadas -> {out_path}")


01_login.png: 23 cajas dibujadas -> vision_evaluation\omniparser\01_login__todas_las_cajas.png


02_alta_cliente.png: 25 cajas dibujadas -> vision_evaluation\omniparser\02_alta_cliente__todas_las_cajas.png


03_popup_validacion.png: 26 cajas dibujadas -> vision_evaluation\omniparser\03_popup_validacion__todas_las_cajas.png


In [11]:
TARGET_OMNIPARSER_KEYWORD = {
    "usuario_field": "usuario",
    "contraseña_field": "contras",
    "entrar_button": "entrar",
    "nombre_field": "nombre",
    "nif_field": "nif",
    "anadir_button": "anadir",
    "popup_aceptar_button": "aceptar",
}


def match_omniparser_target(elementos: list[dict], keyword: str):
    """Busca el primer elemento cuyo caption normalizado (via `_normalize`,
    ya definida en la seccion de RapidOCR) contiene `keyword`, y devuelve
    el centro de su bbox_px, o None si no aparece ninguno."""
    for el in elementos:
        if keyword in _normalize(el["caption"]):
            x1, y1, x2, y2 = el["bbox_px"]
            return (x1 + x2) / 2, (y1 + y2) / 2
    return None


filas_omniparser = []
for image_name, target_key, _desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]

    if image_name not in _omniparser_cache:
        _omniparser_cache[image_name] = ask_omniparser_elements(image_name)
    resultado = _omniparser_cache[image_name]
    elementos = resultado["raw"]
    tiempo = resultado["elapsed_s"]

    keyword = TARGET_OMNIPARSER_KEYWORD[target_key]
    punto = match_omniparser_target(elementos, keyword)
    pred_x = pred_y = error_px = None
    if punto is not None:
        pred_x, pred_y = punto
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    else:
        estado = f"ERROR: ningun caption contiene '{keyword}'"

    filas_omniparser.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px"
    )

    # circulo verde = posicion real, rojo = punto que da OmniParser via caption
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR_OMNIPARSER / f"{_Path(image_name).stem}__{target_key}.png")

df_omniparser = pd.DataFrame(filas_omniparser)
aciertos = (df_omniparser["error_px"] < 20).sum()
print(f"\nResumen omniparser: {aciertos}/{len(df_omniparser)} aciertos (error < 20px)")
print(f"error medio: {df_omniparser['error_px'].mean():.1f}px")
print(f"tiempo medio (deteccion+caption por imagen): {df_omniparser['tiempo_s'].mean():.2f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR_OMNIPARSER}/")
df_omniparser


01_login.png/usuario_field: ERROR: ningun caption contiene 'usuario'  gt=(272,138) pred=(None,None) error=Nonepx
01_login.png/contraseña_field: ERROR: ningun caption contiene 'contras'  gt=(272,188) pred=(None,None) error=Nonepx
01_login.png/entrar_button: ERROR: ningun caption contiene 'entrar'  gt=(208,274) pred=(None,None) error=Nonepx
02_alta_cliente.png/nombre_field: ERROR: ningun caption contiene 'nombre'  gt=(219,1399) pred=(None,None) error=Nonepx
02_alta_cliente.png/nif_field: ERROR: ningun caption contiene 'nif'  gt=(525,1399) pred=(None,None) error=Nonepx
02_alta_cliente.png/anadir_button: ERROR: ningun caption contiene 'anadir'  gt=(475,1483) pred=(None,None) error=Nonepx


03_popup_validacion.png/popup_aceptar_button: ERROR: ningun caption contiene 'aceptar'  gt=(1397,888) pred=(None,None) error=Nonepx

Resumen omniparser: 0/7 aciertos (error < 20px)
error medio: nanpx
tiempo medio (deteccion+caption por imagen): 0.14s
capturas con circulos guardadas en: vision_evaluation\omniparser/


,imagen,target,gt_x,gt_y,pred_x,pred_y,error_px,tiempo_s,estado
0,01_login.png,usuario_field,272,138,None,None,None,0.149276,ERROR: ningun caption contiene 'usuario'
1,01_login.png,contraseña_field,272,188,None,None,None,0.149276,ERROR: ningun caption contiene 'contras'
2,01_login.png,entrar_button,208,274,None,None,None,0.149276,ERROR: ningun caption contiene 'entrar'
3,02_alta_cliente.png,nombre_field,219,1399,None,None,None,0.138420,ERROR: ningun caption contiene 'nombre'
4,02_alta_cliente.png,nif_field,525,1399,None,None,None,0.138420,ERROR: ningun caption contiene 'nif'
5,02_alta_cliente.png,anadir_button,475,1483,None,None,None,0.138420,ERROR: ningun caption contiene 'anadir'
6,03_popup_validacion.png,popup_aceptar_button,1397,888,None,None,None,0.141256,ERROR: ningun caption contiene 'aceptar'


## 3. UI-TARS (Hugging Face `transformers`)

`bytedance-research/UI-TARS-7B-DPO`, ~16 GB en bf16, misma arquitectura que Qwen2-VL. Ya ejecutado y descargado.

Esta pensado especificamente para agentes de GUI: espera que le pidas una accion (`click`) y responde con su propio formato de accion (`click(start_box='(x,y)')`), del que sacamos las coordenadas.

**Resultado real**: muy buenos resultados. La clave estaba en el reescalado -- UI-TARS-7B/2B (basados en Qwen2-VL, no en Qwen2.5-VL) devuelven las coordenadas en una escala relativa fija de 0 a 1000 sobre la imagen ORIGINAL (`x_absoluto = x_relativo / 1000 * ancho_imagen`), segun la [guia oficial del repo](https://github.com/bytedance/UI-TARS/blob/main/README_v1.md#explanation-of-inference-results) -- nada de reconstruir ningun tamaño interno redimensionado, a diferencia de qwen2.5vl (ver nota de descarte al principio del notebook).

In [7]:
_uitars_model = None
_uitars_processor = None


def _load_uitars():
    global _uitars_model, _uitars_processor
    if _uitars_model is None:
        from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

        model_id = "ByteDance-Seed/UI-TARS-7B-DPO"
        _uitars_processor = AutoProcessor.from_pretrained(model_id)
        _uitars_model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, device_map="auto"
        )
    return _uitars_model, _uitars_processor


def ask_uitars(image_name: str, target_desc: str) -> dict:
    model, processor = _load_uitars()
    image = Image.open(GT_DIR / image_name).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": f"Click on {target_desc}. Respond with click(start_box='(x,y)').",
                },
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    output_ids = model.generate(**inputs, max_new_tokens=128)
    elapsed = time.perf_counter() - t0

    raw = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    return {"raw": raw, "elapsed_s": elapsed}


In [ ]:
# prueba rapida con un solo target: dispara la descarga de UI-TARS (~16 GB)
resultado = ask_uitars(TARGETS[2][0], TARGETS[2][2])
print(f"tiempo: {resultado['elapsed_s']:.1f}s\n")
print(resultado["raw"])


In [ ]:
import re

EVAL_OUT_DIR_UITARS = _Path("vision_evaluation") / "uitars"
EVAL_OUT_DIR_UITARS.mkdir(parents=True, exist_ok=True)


def parse_uitars_xy(raw: str):
    """Extrae (x, y) del formato de accion de UI-TARS: click(start_box='(x,y)').

    Segun la guia oficial del repo (README_v1.md, seccion "Explanation of
    Inference Results" / "Coordinate Mapping"), estos numeros NO son pixeles
    de ninguna imagen redimensionada internamente: son una posicion relativa
    en una escala fija de 0 a 1000, pensada para aplicarse directamente sobre
    el ancho/alto de la imagen ORIGINAL:

        x_absoluto = x_relativo / 1000 * ancho_imagen
        y_absoluto = y_relativo / 1000 * alto_imagen

    (Ese convenio de 0-1000 es el de UI-TARS 7B/2B, basados en Qwen2-VL. El
    convenio de "coordenadas absolutas en la imagen redimensionada" que usa
    Qwen2.5-VL -- un modelo distinto, por eso no se uso aqui -- necesitaria
    en cambio reconstruir el tamano exacto que vio el modelo via su propio
    `smart_resize`; con qwen2.5vl via Ollama no podiamos recuperar ese dato
    con certeza, por eso se descarto esa via en vez de intentar arreglarla.)
    """
    m = re.search(r"start_box='?\(?\s*(-?\d+)\s*,\s*(-?\d+)\s*\)?'?", raw)
    if m:
        return float(m.group(1)), float(m.group(2))
    nums = re.findall(r"-?\d+\.?\d*", raw)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    raise ValueError(f"no pude extraer coordenadas de: {raw!r}")


filas_uitars = []
for image_name, target_key, target_desc in TARGETS:
    gt_x, gt_y = GT[image_name][target_key]
    width, height = Image.open(GT_DIR / image_name).size
    pred_x = pred_y = error_px = tiempo = None
    try:
        resultado = ask_uitars(image_name, target_desc)
        tiempo = resultado["elapsed_s"]
        rel_x, rel_y = parse_uitars_xy(resultado["raw"])
        pred_x = rel_x / 1000 * width
        pred_y = rel_y / 1000 * height
        error_px = ((pred_x - gt_x) ** 2 + (pred_y - gt_y) ** 2) ** 0.5
        estado = "ok"
    except Exception as exc:
        estado = f"ERROR: {exc}"

    filas_uitars.append(
        {
            "imagen": image_name,
            "target": target_key,
            "gt_x": gt_x,
            "gt_y": gt_y,
            "pred_x": pred_x,
            "pred_y": pred_y,
            "error_px": error_px,
            "tiempo_s": tiempo,
            "estado": estado,
        }
    )
    print(
        f"{image_name}/{target_key}: {estado}  "
        f"gt=({gt_x},{gt_y}) pred=({pred_x},{pred_y}) "
        f"error={None if error_px is None else round(error_px, 1)}px "
        f"({tiempo if tiempo is None else round(tiempo, 1)}s)"
    )

    # circulo verde = posicion real, rojo = prediccion de UI-TARS (ya reescalada)
    img = Image.open(GT_DIR / image_name).convert("RGB")
    draw = ImageDraw.Draw(img)
    r = 18
    draw.ellipse((gt_x - r, gt_y - r, gt_x + r, gt_y + r), outline=(0, 200, 0), width=5)
    if pred_x is not None:
        draw.ellipse((pred_x - r, pred_y - r, pred_x + r, pred_y + r), outline=(255, 0, 0), width=5)
    img.save(EVAL_OUT_DIR_UITARS / f"{_Path(image_name).stem}__{target_key}.png")

df_uitars = pd.DataFrame(filas_uitars)
aciertos = (df_uitars["error_px"] < 20).sum()
print(f"\nResumen UI-TARS: {aciertos}/{len(df_uitars)} aciertos (error < 20px)")
print(f"error medio: {df_uitars['error_px'].mean():.1f}px")
print(f"tiempo medio: {df_uitars['tiempo_s'].mean():.1f}s")
print(f"capturas con circulos guardadas en: {EVAL_OUT_DIR_UITARS}/")
df_uitars
